COAST: Contrastive Conceptor Activation Steering - https://arxiv.org/abs/2605.17144  
Базовая модель: Diffusion Policy https://arxiv.org/abs/2303.04137  
Идея задачи: нужно взять готовую state-based Diffusion Policy на PushT и попробовать адаптировать к ней идею steering через активации. Это небольшая адаптация идеи COAST на более легкий стенд.  

# 1) поднять бейзлайны, запустить state-based Diffusion Policy на PushT, убедиться, что всё работает

In [1]:
%pip install \
    numpy==1.24.4 \
    torch==1.13.1 \
    torchvision==0.14.1 \
    diffusers==0.18.2 \
    huggingface_hub==0.25.2 \
    scikit-image==0.19.3 \
    scikit-video==1.1.11 \
    pygame==2.1.2 \
    pymunk==6.2.1 \
    gym==0.26.2 \
    shapely==1.8.4 \
    zarr==2.18.3 \
    numcodecs==0.13.1 \
    opencv-python-headless==4.10.0.84 \
    imageio \
    imageio-ffmpeg \
    gdown \
    pandas==2.2.3 \
    tqdm \
    ipykernel

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import sys
import glob
import json
import math
import pickle
import collections
import gc
from typing import Tuple, Sequence, Dict, Union, Optional

import numpy as np
import pandas as pd
import skimage
import skimage.transform as st
import shapely
import shapely.geometry as sg
import cv2

import torch
import torch.nn as nn
import diffusers
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler

import gym
from gym import spaces
import pygame
import pymunk
import pymunk.pygame_util
from pymunk.space_debug_draw_options import SpaceDebugColor
from pymunk.vec2d import Vec2d

from tqdm.auto import tqdm
import imageio.v2 as imageio
from IPython.display import Video, display, Markdown
import gdown
import zarr
import numcodecs

import numpy.core.multiarray
import numpy.core.numeric

sys.modules['numpy._core'] = np.core
sys.modules['numpy._core.multiarray'] = np.core.multiarray
sys.modules['numpy._core.numeric'] = np.core.numeric

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("scikit-image:", skimage.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 1.13.1+cu117
NumPy: 1.24.4
scikit-image: 0.19.3
CUDA available: True
GPU: NVIDIA A10


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
save_dir = '/home/ubuntu/COAST_PushT'

stats_data = np.load(os.path.join(save_dir, 'normalization_stats.npz'))
stats = {'obs': {'min': stats_data['obs_min'], 'max': stats_data['obs_max']}, 'action': {'min': stats_data['action_min'], 'max': stats_data['action_max']}}
print('Normalization statistics loaded')

Normalization statistics loaded


In [4]:
#@markdown ### **Environment**
#@markdown Defines a PyMunk-based Push-T environment `PushTEnv`.
#@markdown
#@markdown **Goal**: push the gray T-block into the green area.
#@markdown
#@markdown Adapted from [Implicit Behavior Cloning](https://implicitbc.github.io/)


positive_y_is_up: bool = False

def to_pygame(p: Tuple[float, float], surface: pygame.Surface) -> Tuple[int, int]:
    if positive_y_is_up:
        return round(p[0]), surface.get_height() - round(p[1])
    else:
        return round(p[0]), round(p[1])


def light_color(color: SpaceDebugColor):
    color = np.minimum(1.2 * np.float32([color.r, color.g, color.b, color.a]), np.float32([255]))
    color = SpaceDebugColor(r=color[0], g=color[1], b=color[2], a=color[3])
    return color

class DrawOptions(pymunk.SpaceDebugDrawOptions):
    def __init__(self, surface: pygame.Surface) -> None:
        self.surface = surface
        super(DrawOptions, self).__init__()

    def draw_circle(
        self,
        pos: Vec2d,
        angle: float,
        radius: float,
        outline_color: SpaceDebugColor,
        fill_color: SpaceDebugColor,
    ) -> None:
        p = to_pygame(pos, self.surface)

        pygame.draw.circle(self.surface, fill_color.as_int(), p, round(radius), 0)
        pygame.draw.circle(self.surface, light_color(fill_color).as_int(), p, round(radius-4), 0)

        circle_edge = pos + Vec2d(radius, 0).rotated(angle)
        p2 = to_pygame(circle_edge, self.surface)
        line_r = 2 if radius > 20 else 1
        # pygame.draw.lines(self.surface, outline_color.as_int(), False, [p, p2], line_r)

    def draw_segment(self, a: Vec2d, b: Vec2d, color: SpaceDebugColor) -> None:
        p1 = to_pygame(a, self.surface)
        p2 = to_pygame(b, self.surface)

        pygame.draw.aalines(self.surface, color.as_int(), False, [p1, p2])

    def draw_fat_segment(
        self,
        a: Tuple[float, float],
        b: Tuple[float, float],
        radius: float,
        outline_color: SpaceDebugColor,
        fill_color: SpaceDebugColor,
    ) -> None:
        p1 = to_pygame(a, self.surface)
        p2 = to_pygame(b, self.surface)

        r = round(max(1, radius * 2))
        pygame.draw.lines(self.surface, fill_color.as_int(), False, [p1, p2], r)
        if r > 2:
            orthog = [abs(p2[1] - p1[1]), abs(p2[0] - p1[0])]
            if orthog[0] == 0 and orthog[1] == 0:
                return
            scale = radius / (orthog[0] * orthog[0] + orthog[1] * orthog[1]) ** 0.5
            orthog[0] = round(orthog[0] * scale)
            orthog[1] = round(orthog[1] * scale)
            points = [
                (p1[0] - orthog[0], p1[1] - orthog[1]),
                (p1[0] + orthog[0], p1[1] + orthog[1]),
                (p2[0] + orthog[0], p2[1] + orthog[1]),
                (p2[0] - orthog[0], p2[1] - orthog[1]),
            ]
            pygame.draw.polygon(self.surface, fill_color.as_int(), points)
            pygame.draw.circle(
                self.surface,
                fill_color.as_int(),
                (round(p1[0]), round(p1[1])),
                round(radius),
            )
            pygame.draw.circle(
                self.surface,
                fill_color.as_int(),
                (round(p2[0]), round(p2[1])),
                round(radius),
            )

    def draw_polygon(
        self,
        verts: Sequence[Tuple[float, float]],
        radius: float,
        outline_color: SpaceDebugColor,
        fill_color: SpaceDebugColor,
    ) -> None:
        ps = [to_pygame(v, self.surface) for v in verts]
        ps += [ps[0]]

        radius = 2
        pygame.draw.polygon(self.surface, light_color(fill_color).as_int(), ps)

        if radius > 0:
            for i in range(len(verts)):
                a = verts[i]
                b = verts[(i + 1) % len(verts)]
                self.draw_fat_segment(a, b, radius, fill_color, fill_color)

    def draw_dot(
        self, size: float, pos: Tuple[float, float], color: SpaceDebugColor
    ) -> None:
        p = to_pygame(pos, self.surface)
        pygame.draw.circle(self.surface, color.as_int(), p, round(size), 0)


def pymunk_to_shapely(body, shapes):
    geoms = list()
    for shape in shapes:
        if isinstance(shape, pymunk.shapes.Poly):
            verts = [body.local_to_world(v) for v in shape.get_vertices()]
            verts += [verts[0]]
            geoms.append(sg.Polygon(verts))
        else:
            raise RuntimeError(f'Unsupported shape type {type(shape)}')
    geom = sg.MultiPolygon(geoms)
    return geom

# env
class PushTEnv(gym.Env):
    metadata = {"render.modes": ["human", "rgb_array"], "video.frames_per_second": 10}
    reward_range = (0., 1.)

    def __init__(self,
            legacy=False,
            block_cog=None, damping=None,
            render_action=True,
            render_size=96,
            reset_to_state=None
        ):
        self._seed = None
        self.seed()
        self.window_size = ws = 512  # The size of the PyGame window
        self.render_size = render_size
        self.sim_hz = 100
        # Local controller params.
        self.k_p, self.k_v = 100, 20    # PD control.z
        self.control_hz = self.metadata['video.frames_per_second']
        # legcay set_state for data compatiblity
        self.legacy = legacy

        # agent_pos, block_pos, block_angle
        self.observation_space = spaces.Box(
            low=np.array([0,0,0,0,0], dtype=np.float64),
            high=np.array([ws,ws,ws,ws,np.pi*2], dtype=np.float64),
            shape=(5,),
            dtype=np.float64
        )

        # positional goal for agent
        self.action_space = spaces.Box(
            low=np.array([0,0], dtype=np.float64),
            high=np.array([ws,ws], dtype=np.float64),
            shape=(2,),
            dtype=np.float64
        )

        self.block_cog = block_cog
        self.damping = damping
        self.render_action = render_action

        """
        If human-rendering is used, `self.window` will be a reference
        to the window that we draw to. `self.clock` will be a clock that is used
        to ensure that the environment is rendered at the correct framerate in
        human-mode. They will remain `None` until human-mode is used for the
        first time.
        """
        self.window = None
        self.clock = None
        self.screen = None

        self.space = None
        self.teleop = None
        self.render_buffer = None
        self.latest_action = None
        self.reset_to_state = reset_to_state

    def reset(self):
        seed = self._seed
        self._setup()
        if self.block_cog is not None:
            self.block.center_of_gravity = self.block_cog
        if self.damping is not None:
            self.space.damping = self.damping

        # use legacy RandomState for compatiblity
        state = self.reset_to_state
        if state is None:
            rs = np.random.RandomState(seed=seed)
            state = np.array([
                rs.randint(50, 450), rs.randint(50, 450),
                rs.randint(100, 400), rs.randint(100, 400),
                rs.randn() * 2 * np.pi - np.pi
                ])
        self._set_state(state)

        obs = self._get_obs()
        info = self._get_info()
        return obs, info

    def step(self, action):
        dt = 1.0 / self.sim_hz
        self.n_contact_points = 0
        n_steps = self.sim_hz // self.control_hz
        if action is not None:
            self.latest_action = action
            for i in range(n_steps):
                # Step PD control.
                # self.agent.velocity = self.k_p * (act - self.agent.position)    # P control works too.
                acceleration = self.k_p * (action - self.agent.position) + self.k_v * (Vec2d(0, 0) - self.agent.velocity)
                self.agent.velocity += acceleration * dt

                # Step physics.
                self.space.step(dt)

        # compute reward
        goal_body = self._get_goal_pose_body(self.goal_pose)
        goal_geom = pymunk_to_shapely(goal_body, self.block.shapes)
        block_geom = pymunk_to_shapely(self.block, self.block.shapes)

        intersection_area = goal_geom.intersection(block_geom).area
        goal_area = goal_geom.area
        coverage = intersection_area / goal_area
        reward = np.clip(coverage / self.success_threshold, 0, 1)
        done = coverage > self.success_threshold
        terminated = done
        truncated = False

        observation = self._get_obs()
        info = self._get_info()
        info['coverage'] = coverage # Save raw geometric coverage for evaluation

        return observation, reward, terminated, truncated, info

    def render(self, mode):
        return self._render_frame(mode)

    def teleop_agent(self):
        TeleopAgent = collections.namedtuple('TeleopAgent', ['act'])
        def act(obs):
            act = None
            mouse_position = pymunk.pygame_util.from_pygame(Vec2d(*pygame.mouse.get_pos()), self.screen)
            if self.teleop or (mouse_position - self.agent.position).length < 30:
                self.teleop = True
                act = mouse_position
            return act
        return TeleopAgent(act)

    def _get_obs(self):
        obs = np.array(
            tuple(self.agent.position) \
            + tuple(self.block.position) \
            + (self.block.angle % (2 * np.pi),))
        return obs

    def _get_goal_pose_body(self, pose):
        mass = 1
        inertia = pymunk.moment_for_box(mass, (50, 100))
        body = pymunk.Body(mass, inertia)
        # preserving the legacy assignment order for compatibility
        # the order here dosn't matter somehow, maybe because CoM is aligned with body origin
        body.position = pose[:2].tolist()
        body.angle = pose[2]
        return body

    def _get_info(self):
        n_steps = self.sim_hz // self.control_hz
        n_contact_points_per_step = int(np.ceil(self.n_contact_points / n_steps))
        info = {
            'pos_agent': np.array(self.agent.position),
            'vel_agent': np.array(self.agent.velocity),
            'block_pose': np.array(list(self.block.position) + [self.block.angle]),
            'goal_pose': self.goal_pose,
            'n_contacts': n_contact_points_per_step}
        return info

    def _render_frame(self, mode):

        if self.window is None and mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
        if self.clock is None and mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        self.screen = canvas

        draw_options = DrawOptions(canvas)

        # Draw goal pose.
        goal_body = self._get_goal_pose_body(self.goal_pose)
        for shape in self.block.shapes:
            goal_points = [pymunk.pygame_util.to_pygame(goal_body.local_to_world(v), draw_options.surface) for v in shape.get_vertices()]
            goal_points += [goal_points[0]]
            pygame.draw.polygon(canvas, self.goal_color, goal_points)

        # Draw agent and block.
        self.space.debug_draw(draw_options)

        if mode == "human":
            # The following line copies our drawings from `canvas` to the visible window
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()

            # the clock is aleady ticked during in step for "human"


        img = np.transpose(
                np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2)
            )
        img = cv2.resize(img, (self.render_size, self.render_size))
        if self.render_action:
            if self.render_action and (self.latest_action is not None):
                action = np.array(self.latest_action)
                coord = (action / 512 * 96).astype(np.int32)
                marker_size = int(8/96*self.render_size)
                thickness = int(1/96*self.render_size)
                cv2.drawMarker(img, coord,
                    color=(255,0,0), markerType=cv2.MARKER_CROSS,
                    markerSize=marker_size, thickness=thickness)
        return img


    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()

    def seed(self, seed=None):
        if seed is None:
            seed = np.random.randint(0,25536)
        self._seed = seed
        self.np_random = np.random.default_rng(seed)

    def _handle_collision(self, arbiter, space, data):
        self.n_contact_points += len(arbiter.contact_point_set.points)

    def _set_state(self, state):
        if isinstance(state, np.ndarray):
            state = state.tolist()
        pos_agent = state[:2]
        pos_block = state[2:4]
        rot_block = state[4]
        self.agent.position = pos_agent
        # setting angle rotates with respect to center of mass
        # therefore will modify the geometric position
        # if not the same as CoM
        # therefore should be modified first.
        if self.legacy:
            # for compatiblity with legacy data
            self.block.position = pos_block
            self.block.angle = rot_block
        else:
            self.block.angle = rot_block
            self.block.position = pos_block

        # Run physics to take effect
        self.space.step(1.0 / self.sim_hz)

    def _set_state_local(self, state_local):
        agent_pos_local = state_local[:2]
        block_pose_local = state_local[2:]
        tf_img_obj = st.AffineTransform(
            translation=self.goal_pose[:2],
            rotation=self.goal_pose[2])
        tf_obj_new = st.AffineTransform(
            translation=block_pose_local[:2],
            rotation=block_pose_local[2]
        )
        tf_img_new = st.AffineTransform(
            matrix=tf_img_obj.params @ tf_obj_new.params
        )
        agent_pos_new = tf_img_new(agent_pos_local)
        new_state = np.array(
            list(agent_pos_new[0]) + list(tf_img_new.translation) \
                + [tf_img_new.rotation])
        self._set_state(new_state)
        return new_state

    def _setup(self):
        self.space = pymunk.Space()
        self.space.gravity = 0, 0
        self.space.damping = 0
        self.teleop = False
        self.render_buffer = list()

        # Add walls.
        walls = [
            self._add_segment((5, 506), (5, 5), 2),
            self._add_segment((5, 5), (506, 5), 2),
            self._add_segment((506, 5), (506, 506), 2),
            self._add_segment((5, 506), (506, 506), 2)
        ]
        self.space.add(*walls)

        # Add agent, block, and goal zone.
        self.agent = self.add_circle((256, 400), 15)
        self.block = self.add_tee((256, 300), 0)
        self.goal_color = pygame.Color('LightGreen')
        self.goal_pose = np.array([256,256,np.pi/4])  # x, y, theta (in radians)

        # Add collision handeling
        self.collision_handeler = self.space.add_collision_handler(0, 0)
        self.collision_handeler.post_solve = self._handle_collision
        self.n_contact_points = 0

        self.max_score = 50 * 100
        self.success_threshold = 0.95    # 95% coverage.

    def _add_segment(self, a, b, radius):
        shape = pymunk.Segment(self.space.static_body, a, b, radius)
        shape.color = pygame.Color('LightGray')    # https://htmlcolorcodes.com/color-names
        return shape

    def add_circle(self, position, radius):
        body = pymunk.Body(body_type=pymunk.Body.KINEMATIC)
        body.position = position
        body.friction = 1
        shape = pymunk.Circle(body, radius)
        shape.color = pygame.Color('RoyalBlue')
        self.space.add(body, shape)
        return body

    def add_box(self, position, height, width):
        mass = 1
        inertia = pymunk.moment_for_box(mass, (height, width))
        body = pymunk.Body(mass, inertia)
        body.position = position
        shape = pymunk.Poly.create_box(body, (height, width))
        shape.color = pygame.Color('LightSlateGray')
        self.space.add(body, shape)
        return body

    def add_tee(self, position, angle, scale=30, color='LightSlateGray', mask=pymunk.ShapeFilter.ALL_MASKS()):
        mass = 1
        length = 4
        vertices1 = [(-length*scale/2, scale),
                                 ( length*scale/2, scale),
                                 ( length*scale/2, 0),
                                 (-length*scale/2, 0)]
        inertia1 = pymunk.moment_for_poly(mass, vertices=vertices1)
        vertices2 = [(-scale/2, scale),
                                 (-scale/2, length*scale),
                                 ( scale/2, length*scale),
                                 ( scale/2, scale)]
        inertia2 = pymunk.moment_for_poly(mass, vertices=vertices1)
        body = pymunk.Body(mass, inertia1 + inertia2)
        shape1 = pymunk.Poly(body, vertices1)
        shape2 = pymunk.Poly(body, vertices2)
        shape1.color = pygame.Color(color)
        shape2.color = pygame.Color(color)
        shape1.filter = pymunk.ShapeFilter(mask=mask)
        shape2.filter = pymunk.ShapeFilter(mask=mask)
        body.center_of_gravity = (shape1.center_of_gravity + shape2.center_of_gravity) / 2
        body.position = position
        body.angle = angle
        body.friction = 1
        self.space.add(body, shape1, shape2)
        return body

In [5]:
#@markdown ### **Env Demo**
#@markdown Standard Gym Env (0.21.0 API)

# 0. create env object
env = PushTEnv()

# 1. seed env for initial state.
# Seed 0-200 are used for the demonstration dataset.
env.seed(1000)

# 2. must reset before use
obs, info = env.reset()

# 3. 2D positional action space [0,512]
action = env.action_space.sample()

# 4. Standard gym step method
obs, reward, terminated, truncated, info = env.step(action)

# prints and explains each dimension of the observation and action vectors
with np.printoptions(precision=4, suppress=True, threshold=5):
    print("Obs: ", repr(obs))
    print("Obs:        [agent_x,  agent_y,  block_x,  block_y,    block_angle]")
    print("Action: ", repr(action))
    print("Action:   [target_agent_x, target_agent_y]")
    print("Coverage:", info['coverage'])

Obs:  array([163.3562, 230.112 , 292.    , 351.    ,   2.9196])
Obs:        [agent_x,  agent_y,  block_x,  block_y,    block_angle]
Action:  array([226.1401, 490.0308])
Action:   [target_agent_x, target_agent_y]
Coverage: 0.17837702012988743


In [6]:
#@markdown ### **Dataset**
#@markdown
#@markdown Defines `PushTStateDataset` and helper functions
#@markdown
#@markdown The dataset class
#@markdown - Load data (obs, action) from a zarr storage
#@markdown - Normalizes each dimension of obs and action to [-1,1]
#@markdown - Returns
#@markdown  - All possible segments with length `pred_horizon`
#@markdown  - Pads the beginning and the end of each episode with repetition
#@markdown  - key `obs`: shape (obs_horizon, obs_dim)
#@markdown  - key `action`: shape (pred_horizon, action_dim)

# normalize data
def get_data_stats(data):
    data = data.reshape(-1,data.shape[-1])
    stats = {
        'min': np.min(data, axis=0),
        'max': np.max(data, axis=0)
    }
    return stats

def normalize_data(data, stats):
    # nomalize to [0,1]
    ndata = (data - stats['min']) / (stats['max'] - stats['min'])
    # normalize to [-1, 1]
    ndata = ndata * 2 - 1
    return ndata

def unnormalize_data(ndata, stats):
    ndata = (ndata + 1) / 2
    data = ndata * (stats['max'] - stats['min']) + stats['min']
    return data


In [7]:
#@markdown ### **Dataset Demo**

# download demonstration data from Google Drive
dataset_path = "pusht_cchi_v7_replay.zarr.zip"
if not os.path.isfile(dataset_path):
    id = "1KY1InLurpMvJDRb14L9NlXT_fEsCvVUq&confirm=t"
    gdown.download(id=id, output=dataset_path, quiet=False)

# parameters
pred_horizon = 16
obs_horizon = 2
action_horizon = 8

# Open dataset
dataset_root = zarr.open(dataset_path, mode='r')

# Calculate normalization statistics
stats = {'obs': get_data_stats(dataset_root['data']['state'][:]),
    'action': get_data_stats(dataset_root['data']['action'][:])}

print("Observation stats:")
print("min:", stats['obs']['min'])
print("max:", stats['obs']['max'])

print("\nAction stats:")
print("min:", stats['action']['min'])
print("max:", stats['action']['max'])


Observation stats:
min: [1.3456424e+01 3.2938293e+01 5.7471767e+01 1.0827995e+02 2.1559125e-04]
max: [496.14618   510.9579    439.9153    485.6641      6.2830877]

Action stats:
min: [12. 25.]
max: [511. 511.]


In [8]:
#@markdown ### **Network**
#@markdown
#@markdown Defines a 1D UNet architecture `ConditionalUnet1D`
#@markdown as the noies prediction network
#@markdown
#@markdown Components
#@markdown - `SinusoidalPosEmb` Positional encoding for the diffusion iteration k
#@markdown - `Downsample1d` Strided convolution to reduce temporal resolution
#@markdown - `Upsample1d` Transposed convolution to increase temporal resolution
#@markdown - `Conv1dBlock` Conv1d --> GroupNorm --> Mish
#@markdown - `ConditionalResidualBlock1D` Takes two inputs `x` and `cond`. \
#@markdown `x` is passed through 2 `Conv1dBlock` stacked together with residual connection.
#@markdown `cond` is applied to `x` with [FiLM](https://arxiv.org/abs/1709.07871) conditioning.

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb


class Downsample1d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv1d(dim, dim, 3, 2, 1)

    def forward(self, x):
        return self.conv(x)

class Upsample1d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.ConvTranspose1d(dim, dim, 4, 2, 1)

    def forward(self, x):
        return self.conv(x)


class Conv1dBlock(nn.Module):
    '''
        Conv1d --> GroupNorm --> Mish
    '''

    def __init__(self, inp_channels, out_channels, kernel_size, n_groups=8):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv1d(inp_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.GroupNorm(n_groups, out_channels),
            nn.Mish(),
        )

    def forward(self, x):
        return self.block(x)


class ConditionalResidualBlock1D(nn.Module):
    def __init__(self,
            in_channels,
            out_channels,
            cond_dim,
            kernel_size=3,
            n_groups=8):
        super().__init__()

        self.blocks = nn.ModuleList([
            Conv1dBlock(in_channels, out_channels, kernel_size, n_groups=n_groups),
            Conv1dBlock(out_channels, out_channels, kernel_size, n_groups=n_groups),
        ])

        # FiLM modulation https://arxiv.org/abs/1709.07871
        # predicts per-channel scale and bias
        cond_channels = out_channels * 2
        self.out_channels = out_channels
        self.cond_encoder = nn.Sequential(
            nn.Mish(),
            nn.Linear(cond_dim, cond_channels),
            nn.Unflatten(-1, (-1, 1))
        )

        # make sure dimensions compatible
        self.residual_conv = nn.Conv1d(in_channels, out_channels, 1) \
            if in_channels != out_channels else nn.Identity()

    def forward(self, x, cond):
        '''
            x : [ batch_size x in_channels x horizon ]
            cond : [ batch_size x cond_dim]

            returns:
            out : [ batch_size x out_channels x horizon ]
        '''
        out = self.blocks[0](x)
        embed = self.cond_encoder(cond)

        embed = embed.reshape(
            embed.shape[0], 2, self.out_channels, 1)
        scale = embed[:,0,...]
        bias = embed[:,1,...]
        out = scale * out + bias

        out = self.blocks[1](out)
        out = out + self.residual_conv(x)
        return out


class ConditionalUnet1D(nn.Module):
    def __init__(self,
        input_dim,
        global_cond_dim,
        diffusion_step_embed_dim=256,
        down_dims=[256,512,1024],
        kernel_size=5,
        n_groups=8
        ):
        """
        input_dim: Dim of actions.
        global_cond_dim: Dim of global conditioning applied with FiLM
          in addition to diffusion step embedding. This is usually obs_horizon * obs_dim
        diffusion_step_embed_dim: Size of positional encoding for diffusion iteration k
        down_dims: Channel size for each UNet level.
          The length of this array determines numebr of levels.
        kernel_size: Conv kernel size
        n_groups: Number of groups for GroupNorm
        """

        super().__init__()
        all_dims = [input_dim] + list(down_dims)
        start_dim = down_dims[0]

        dsed = diffusion_step_embed_dim
        diffusion_step_encoder = nn.Sequential(
            SinusoidalPosEmb(dsed),
            nn.Linear(dsed, dsed * 4),
            nn.Mish(),
            nn.Linear(dsed * 4, dsed),
        )
        cond_dim = dsed + global_cond_dim

        in_out = list(zip(all_dims[:-1], all_dims[1:]))
        mid_dim = all_dims[-1]
        self.mid_modules = nn.ModuleList([
            ConditionalResidualBlock1D(
                mid_dim, mid_dim, cond_dim=cond_dim,
                kernel_size=kernel_size, n_groups=n_groups
            ),
            ConditionalResidualBlock1D(
                mid_dim, mid_dim, cond_dim=cond_dim,
                kernel_size=kernel_size, n_groups=n_groups
            ),
        ])

        down_modules = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(in_out):
            is_last = ind >= (len(in_out) - 1)
            down_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(
                    dim_in, dim_out, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                ConditionalResidualBlock1D(
                    dim_out, dim_out, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                Downsample1d(dim_out) if not is_last else nn.Identity()
            ]))

        up_modules = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(reversed(in_out[1:])):
            is_last = ind >= (len(in_out) - 1)
            up_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(
                    dim_out*2, dim_in, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                ConditionalResidualBlock1D(
                    dim_in, dim_in, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                Upsample1d(dim_in) if not is_last else nn.Identity()
            ]))

        final_conv = nn.Sequential(
            Conv1dBlock(start_dim, start_dim, kernel_size=kernel_size),
            nn.Conv1d(start_dim, input_dim, 1),
        )

        self.diffusion_step_encoder = diffusion_step_encoder
        self.up_modules = up_modules
        self.down_modules = down_modules
        self.final_conv = final_conv

        print("number of parameters: {:e}".format(
            sum(p.numel() for p in self.parameters()))
        )

    def forward(self,
            sample: torch.Tensor,
            timestep: Union[torch.Tensor, float, int],
            global_cond=None):
        """
        x: (B,T,input_dim)
        timestep: (B,) or int, diffusion step
        global_cond: (B,global_cond_dim)
        output: (B,T,input_dim)
        """
        # (B,T,C)
        sample = sample.moveaxis(-1,-2)
        # (B,C,T)

        # 1. time
        timesteps = timestep
        if not torch.is_tensor(timesteps):
            # TODO: this requires sync between CPU and GPU. So try to pass timesteps as tensors if you can
            timesteps = torch.tensor([timesteps], dtype=torch.long, device=sample.device)
        elif torch.is_tensor(timesteps) and len(timesteps.shape) == 0:
            timesteps = timesteps[None].to(sample.device)
        # broadcast to batch dimension in a way that's compatible with ONNX/Core ML
        timesteps = timesteps.expand(sample.shape[0])

        global_feature = self.diffusion_step_encoder(timesteps)

        if global_cond is not None:
            global_feature = torch.cat([
                global_feature, global_cond
            ], axis=-1)

        x = sample
        h = []
        for idx, (resnet, resnet2, downsample) in enumerate(self.down_modules):
            x = resnet(x, global_feature)
            x = resnet2(x, global_feature)
            h.append(x)
            x = downsample(x)

        for mid_module in self.mid_modules:
            x = mid_module(x, global_feature)

        for idx, (resnet, resnet2, upsample) in enumerate(self.up_modules):
            x = torch.cat((x, h.pop()), dim=1)
            x = resnet(x, global_feature)
            x = resnet2(x, global_feature)
            x = upsample(x)

        x = self.final_conv(x)

        # (B,C,T)
        x = x.moveaxis(-1,-2)
        # (B,T,C)
        return x

print(ConditionalUnet1D)
print(ConditionalResidualBlock1D)

<class '__main__.ConditionalUnet1D'>
<class '__main__.ConditionalResidualBlock1D'>


In [9]:
#@markdown ### **Network Demo**

# observation and action dimensions corrsponding to
# the output of PushTEnv
obs_dim = 5
action_dim = 2

# create network object
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon
)

# example inputs
noised_action = torch.randn((1, pred_horizon, action_dim))
obs = torch.zeros((1, obs_horizon, obs_dim))
diffusion_iter = torch.zeros((1,))

# the noise prediction network
# takes noisy action, diffusion iteration and observation as input
# predicts the noise added to action
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs.flatten(start_dim=1))

# illustration of removing noise
# the actual noise removal is performed by NoiseScheduler
# and is dependent on the diffusion noise schedule
denoised_action = noised_action - noise

# for this demo, we use DDPMScheduler with 100 diffusion iterations
num_diffusion_iters = 100
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

# device transfer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_ = noise_pred_net.to(device)

number of parameters: 6.535322e+07


In [10]:
#@markdown ### **Loading Pretrained Checkpoint**
#@markdown Set `load_pretrained = True` to load pretrained weights.

load_pretrained = True
if load_pretrained:
  ckpt_path = "pusht_state_100ep.ckpt"
  if not os.path.isfile(ckpt_path):
      id = "1mHDr_DEZSdiGo9yecL50BBQYzR8Fjhl_&confirm=t"
      gdown.download(id=id, output=ckpt_path, quiet=False)

  state_dict = torch.load(ckpt_path, map_location=device)
  ema_noise_pred_net = noise_pred_net
  ema_noise_pred_net.load_state_dict(state_dict)
  ema_noise_pred_net.eval()
  print('Pretrained weights loaded.')
else:
  print("Skipped pretrained weight loading.")

Pretrained weights loaded.


In [11]:
#@markdown ### **Inference**

# limit enviornment interaction to 200 steps before termination
max_steps = 200
seed = 100000
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Improve reproducibility on GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

env = PushTEnv()
env.seed(seed)

# get first observation
obs, info = env.reset()

# keep a queue of last 2 steps of observations
obs_deque = collections.deque(
    [obs] * obs_horizon, maxlen=obs_horizon)
# save visualization and rewards
imgs = [env.render(mode='rgb_array')]
rewards = list()
coverages = list()
steps_to_success = None
done = False
step_idx = 0

with tqdm(total=max_steps, desc="Eval PushTStateEnv") as pbar:
    while not done:
        B = 1
        # stack the last obs_horizon (2) number of observations
        obs_seq = np.stack(obs_deque)
        # normalize observation
        nobs = normalize_data(obs_seq, stats=stats['obs'])
        # device transfer
        nobs = torch.from_numpy(nobs).to(device, dtype=torch.float32)

        # infer action
        with torch.no_grad():
            # reshape observation to (B,obs_horizon*obs_dim)
            obs_cond = nobs.unsqueeze(0).flatten(start_dim=1)

            # initialize action from Guassian noise
            noisy_action = torch.randn(
                (B, pred_horizon, action_dim), device=device)
            naction = noisy_action

            # init scheduler
            noise_scheduler.set_timesteps(num_diffusion_iters)

            for k in noise_scheduler.timesteps:
                # predict noise
                noise_pred = ema_noise_pred_net(
                    sample=naction,
                    timestep=k,
                    global_cond=obs_cond
                )

                # inverse diffusion step (remove noise)
                naction = noise_scheduler.step(
                    model_output=noise_pred,
                    timestep=k,
                    sample=naction
                ).prev_sample

        # unnormalize action
        naction = naction.detach().to('cpu').numpy()
        # (B, pred_horizon, action_dim)
        naction = naction[0]
        action_pred = unnormalize_data(naction, stats=stats['action'])

        # only take action_horizon number of actions
        start = obs_horizon - 1
        end = start + action_horizon
        action = action_pred[start:end,:]
        # (action_horizon, action_dim)

        # execute action_horizon number of steps
        # without replanning
        for i in range(len(action)):
            # stepping env
            obs, reward, terminated, truncated, info = env.step(action[i])
            done = terminated or truncated
            if terminated and steps_to_success is None:
              steps_to_success = step_idx + 1
            # save observations
            obs_deque.append(obs)
            # and reward/vis
            rewards.append(reward)
            coverages.append(info['coverage'])
            imgs.append(env.render(mode='rgb_array'))

            # update progress bar
            step_idx += 1
            pbar.update(1)
            pbar.set_postfix(reward=reward)
            if step_idx >= max_steps:
                done = True
            if done:
                break

print('Max reward:', max(rewards))
print('Max coverage:', max(coverages))

if steps_to_success is not None:
    print('Steps to success:', steps_to_success)
else:
    print('Success not achieved within', max_steps, 'steps')

imageio.mimsave('vis.mp4', imgs, fps=10, macro_block_size=1)
Video('vis.mp4', embed=True, width=256, height=256)

Eval PushTStateEnv: 100%|██████████| 200/200 [00:35<00:00,  5.61it/s, reward=0.897]  


Max reward: 0.9659677535739464
Max coverage: 0.9176693658952491
Success not achieved within 200 steps


In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/COAST_PushT'
os.makedirs(save_dir, exist_ok=True)
shutil.copy('vis.mp4', save_dir)
shutil.copy('pusht_state_100ep.ckpt', save_dir)
shutil.copy('pusht_cchi_v7_replay.zarr.zip', save_dir)
np.savez(f'{save_dir}/stats.npz', obs_min=stats['obs']['min'], obs_max=stats['obs']['max'], action_min=stats['action']['min'], action_max=stats['action']['max'])
np.savez(f'{save_dir}/baseline_results.npz', rewards=np.array(rewards), coverages=np.array(coverages),
         steps_to_success=steps_to_success if steps_to_success is not None else -1,
         max_reward=max(rewards), max_coverage=max(coverages), seed=seed, max_steps=max_steps)
print(f'Files saved to {save_dir}')

# 2) Собрать эпизоды: успешные и неуспешные rollout. На их основе потом будем строить стиринг.

In [12]:
#@markdown ### **Reproducible Noise**

def get_call_seed(episode_seed, call_idx):
    return int(episode_seed + call_idx * 100_003)

In [13]:
# Activation capture for selected U-Net blocks

layer_modules = {'down1': lambda n: n.down_modules[1][1],
    'mid': lambda n: n.mid_modules[1],
    'up0': lambda n: n.up_modules[0][1],
    'up1': lambda n: n.up_modules[1][1],}
layer_names = list(layer_modules.keys())

class ActRecorder:
    def __init__(self, net):
        self.enabled = True
        self.buf = {name: [] for name in layer_names}
        self.handles = [layer_modules[name](net).register_forward_hook(self._make_hook(name)) for name in layer_names]
    def _make_hook(self, name):
        def hook(module, inputs, output):
            if not self.enabled:
                return
            self.buf[name].append(output.detach())
        return hook

    def begin_call(self):
        self.buf = {name: [] for name in layer_names}

    def pop_call(self):
        
        # Stack activations from all denoising steps and transfer them to CPU
        result = {}

        for name, values in self.buf.items():
            result[name] = (torch.stack(values, dim=0).squeeze(1).to(torch.float16).cpu().numpy())
        self.buf = {name: [] for name in layer_names}
        return result

    def remove(self):
        # Remove all registered forward hooks
        for handle in self.handles:
            handle.remove()
        self.handles = []

In [14]:
# Initialize activation recorder

if 'act_recorder' in globals():
    act_recorder.remove()

act_recorder = ActRecorder(ema_noise_pred_net)

In [15]:
#@markdown ### **Rollout Collection**

def run_episode(seed, max_steps=200, steering_config=None):

    # Create a fresh environment for every episode
    env = PushTEnv()
    env.seed(seed)

    # Get initial observation
    obs, info = env.reset()

    # Keep the last obs_horizon observations
    obs_deque = collections.deque([obs] * obs_horizon, maxlen=obs_horizon)

    # Register steering hook if a configuration is provided
    steering_handle = None
    if steering_config is not None:
        method, strength = steering_config

        if method == 'mean':
            hook = make_mean_steering_hook(steering_vector, strength)
        else:
            M = steering_matrices[(method, strength)].to(device=device, dtype=torch.float32)
            hook = make_steering_hook(M)

        target_module = ema_noise_pred_net.up_modules[0][1]
        steering_handle = target_module.register_forward_hook(hook)

    # Keep activation collection only for baseline
    save_activations = steering_config is None
    act_recorder.enabled = save_activations
    
    # Storage for rollout data
    observations = [obs.copy()]
    actions = []
    rewards = []
    coverages = []

    # Storage for policy calls
    call_pred = []
    call_seeds = []
    call_obs_indices = []

    # Storage for activations
    episode_activations = {name: [] for name in layer_names}

    # Episode metrics
    steps_to_success = None
    done = False
    step_idx = 0
    call_idx = 0

    try:
        while not done:
            B = 1
            call_obs_indices.append([max(step_idx - 1, 0), step_idx])

            # Stack the last obs_horizon observations
            obs_seq = np.stack(obs_deque)

            # Normalize observations
            nobs = normalize_data(obs_seq, stats=stats['obs'])

            # Transfer observations to device
            nobs = torch.from_numpy(nobs).to(device, dtype=torch.float32)

            # Prepare conditioning input
            obs_cond = nobs.unsqueeze(0).flatten(start_dim=1)

            # Create a reproducible random generator for this policy call
            call_seed = get_call_seed(seed, call_idx)
            call_seeds.append(call_seed)
            generator = torch.Generator(device=device)
            generator.manual_seed(call_seed)
            if save_activations:
              act_recorder.begin_call()

            with torch.no_grad():
                # Initialize action from Gaussian noise
                noisy_action = torch.randn((B, pred_horizon, action_dim), device=device,
                    generator=generator)
                naction = noisy_action

                # Initialize scheduler
                noise_scheduler.set_timesteps(num_diffusion_iters)

                for k in noise_scheduler.timesteps:
                    # Predict noise
                    noise_pred = ema_noise_pred_net(sample=naction, timestep=k,
                        global_cond=obs_cond)

                    # Inverse diffusion step
                    naction = noise_scheduler.step(model_output=noise_pred,
                        timestep=k, sample=naction, generator=generator).prev_sample

                # Collect activations only for baseline
                if save_activations:
                    call_activations = act_recorder.pop_call()
                    for name in layer_names:
                        episode_activations[name].append(call_activations[name])

            # Save normalized denoised action prediction
            naction = naction.detach().to('cpu').numpy()
            naction = naction[0]
            call_pred.append(naction.copy())

            # Unnormalize predicted actions
            action_pred = unnormalize_data(naction, stats=stats['action'])

            # Select actions to execute
            start = obs_horizon - 1
            end = start + action_horizon
            action = action_pred[start:end, :]

            # Execute predicted actions
            for i in range(len(action)):
                # Take one environment step
                obs, reward, terminated, truncated, info = env.step(action[i])
                done = terminated or truncated

                # Save success step
                if terminated and steps_to_success is None:
                    steps_to_success = step_idx + 1

                # Save executed action and new observation
                actions.append(action[i].copy())
                observations.append(obs.copy())

                # Save metrics
                rewards.append(reward)
                coverages.append(info['coverage'])

                # Update observation queue
                obs_deque.append(obs)

                # Update step counter
                step_idx += 1

                # Stop if maximum number of steps is reached
                if step_idx >= max_steps:
                    done = True
                if done:
                    break
            call_idx += 1

    finally:
        # Remove steering hook and close environment
        if steering_handle is not None:
            steering_handle.remove()
        env.close()

    # Save activations separately from rollout data (baseline only)
    if save_activations:
        activation_dir = os.path.join(save_dir, 'activations')
        os.makedirs(activation_dir, exist_ok=True)
        activation_path = os.path.join(activation_dir, f'episode_{seed}.npz')
        activation_data = {name: np.stack(values, axis=0)
            for name, values in episode_activations.items()}

        # Save atomically
        temp_path = activation_path + '.tmp'
        with open(temp_path, 'wb') as f:
            np.savez(f, **activation_data)

        os.replace(temp_path, activation_path)

    # Determine episode success
    success = steps_to_success is not None

    # Return collected episode data
    return {
        'seed': seed,
        'success': int(success),
        'steps': step_idx,
        'steps_to_success': steps_to_success if success else -1,

        'obs': np.array(observations),
        'actions': np.array(actions),
        'reward': np.array(rewards),
        'coverage': np.array(coverages),

        'call_pred': np.array(call_pred),
        'call_seeds': np.array(call_seeds),
        'call_obs_indices': np.array(call_obs_indices)}

In [16]:
# Сохранение результатов порциями при обрыве сессий

def collect_chunked(seeds, tag, chunk=10):
    os.makedirs(save_dir, exist_ok=True)

    for i in range(0, len(seeds), chunk):
        path = f'{save_dir}/{tag}_{i:04d}.pkl'
        chunk_seeds = seeds[i:i + chunk]

        activation_dir = os.path.join(save_dir, 'activations')
        activation_files_exist = all(os.path.exists(os.path.join(activation_dir, f'episode_{seed}.npz')) for seed in chunk_seeds)

        if os.path.exists(path) and activation_files_exist:
            print(f'Skipping existing chunk: {path}')
            continue

        print(f'Collecting episodes {i + 1}–{i + len(chunk_seeds)}')
        part = [run_episode(seed) for seed in chunk_seeds]

        temp_path = path + '.tmp'
        with open(temp_path, 'wb') as f:
            pickle.dump(part, f)

        os.replace(temp_path, path)
        print(f'Saved: {path}')
os.makedirs(save_dir, exist_ok=True)

metadata = {'device': str(device), 'torch_version': torch.__version__, 'numpy_version': np.__version__,
            'diffusers_version': diffusers.__version__, 'obs_horizon': obs_horizon, 'pred_horizon': pred_horizon,
            'action_horizon': action_horizon, 'num_diffusion_iters': num_diffusion_iters, 'max_steps': 200,
            'success_threshold': 0.95, 'activation_layers': layer_names, 'activation_dtype': 'float16',
            'activation_storage': 'separate_npz_per_episode', 'activation_pooling': None, 'denoise_order': '99->0',}

with open(f'{save_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

In [34]:
gpu_seeds = list(range(200000, 200020))

collect_chunked(seeds=gpu_seeds, tag='steering_train_act', chunk=5)

Saved: /home/ubuntu/COAST_PushT/steering_train_act_0000.pkl
Saved: /home/ubuntu/COAST_PushT/steering_train_act_0005.pkl
Saved: /home/ubuntu/COAST_PushT/steering_train_act_0010.pkl
Saved: /home/ubuntu/COAST_PushT/steering_train_act_0015.pkl


In [17]:
tag = 'steering_train_act'

chunk_files = sorted([
    f for f in os.listdir(save_dir)
    if f.startswith(tag + '_') and f.endswith('.pkl')])

episodes = []

for filename in chunk_files:
    with open(os.path.join(save_dir, filename), 'rb') as f:
        episodes.extend(pickle.load(f))

print('Total episodes:', len(episodes))
print('Successful:', sum(ep['success'] for ep in episodes))
print('Failed:', sum(not ep['success'] for ep in episodes))

Total episodes: 20
Successful: 5
Failed: 15


In [36]:
# episods for test

gpu_test_seeds = list(range(300000, 300035))
collect_chunked(seeds=gpu_test_seeds, tag='baseline_test', chunk=5)

tag = 'baseline_test'

chunk_files = sorted([
    f for f in os.listdir(save_dir)
    if f.startswith(tag + '_') and f.endswith('.pkl')])

baseline_test_episodes = []

for filename in chunk_files:
    with open(os.path.join(save_dir, filename), 'rb') as f:
        baseline_test_episodes.extend(pickle.load(f))

Saved: /home/ubuntu/COAST_PushT/baseline_test_0000.pkl
Saved: /home/ubuntu/COAST_PushT/baseline_test_0005.pkl
Saved: /home/ubuntu/COAST_PushT/baseline_test_0010.pkl
Saved: /home/ubuntu/COAST_PushT/baseline_test_0015.pkl
Saved: /home/ubuntu/COAST_PushT/baseline_test_0020.pkl
Saved: /home/ubuntu/COAST_PushT/baseline_test_0025.pkl
Saved: /home/ubuntu/COAST_PushT/baseline_test_0030.pkl


In [18]:
baseline_test_episodes = []

test_files = sorted(
    glob.glob(os.path.join(save_dir, 'baseline_test_*.pkl'))
)

for path in test_files:
    with open(path, 'rb') as f:
        baseline_test_episodes.extend(pickle.load(f))

baseline_test_episodes.sort(key=lambda ep: ep['seed'])

print("Всего baseline-эпизодов:", len(baseline_test_episodes))
print("Первый seed:", baseline_test_episodes[0]['seed'])
print("Последний seed:", baseline_test_episodes[-1]['seed'])
print("Успешных:", sum(ep['success'] for ep in baseline_test_episodes))
print("Неуспешных:", sum(not ep['success'] for ep in baseline_test_episodes))

Всего baseline-эпизодов: 35
Первый seed: 300000
Последний seed: 300034
Успешных: 10
Неуспешных: 25


# 3) Выбрать слой / представление: где именно снимать активации у policy.


In [18]:
train_episodes = []
chunk_ids = [0, 5, 10, 15]

for chunk_id in chunk_ids:
    path = os.path.join(save_dir, f'steering_train_act_{chunk_id:04d}.pkl')
    with open(path, 'rb') as f:
        train_episodes.extend(pickle.load(f))

print('Total episodes:', len(train_episodes))

Total episodes: 20


In [19]:
layer_names = ['down1', 'mid', 'up0', 'up1']
alpha = 10.0

def compute_conceptor(X, alpha):
    """X: (N, d), activation vectors."""
    X = np.asarray(X, dtype=np.float64)
    # Mean-centering
    X_centered = X - X.mean(axis=0, keepdims=True)
    # Correlation matrix
    R = (X_centered.T @ X_centered) / len(X_centered)
    # Conceptor
    d = R.shape[0]
    I = np.eye(d, dtype=np.float64)
    C = R @ np.linalg.inv(R + alpha**(-2) * I)
    return C

results = {}

for layer in layer_names:
    success_activations = []
    failure_activations = []

    # Collect activation vectors from all fitting episodes
    for ep in train_episodes:
        seed = ep['seed']
        path = os.path.join(save_dir, 'activations', f'episode_{seed}.npz')
        with np.load(path) as data:
            arr = data[layer]
            # (calls, denoising steps, channels, positions)
            # mean-pool over action positions
            pooled = arr.mean(axis=-1)
            X = pooled.reshape(-1, pooled.shape[-1])
            X = X.astype(np.float64)

        if ep['success']:
            success_activations.append(X)
        else:
            failure_activations.append(X)

    X_success = np.concatenate(success_activations, axis=0)
    X_failure = np.concatenate(failure_activations, axis=0)

    # Fit success and failure conceptors
    C_success = compute_conceptor(X_success, alpha)
    C_failure = compute_conceptor(X_failure, alpha)

    # Boolean AND-NOT
    d = C_success.shape[0]
    I = np.eye(d, dtype=np.float64)
    C_contrastive = np.linalg.pinv(np.linalg.pinv(C_success) + np.linalg.pinv(I - C_failure) - I)

    # Conceptor quota
    quota = np.trace(C_contrastive) / d
    results[layer] = {'quota': quota, 'dimension': d, 'n_success_vectors': len(X_success), 'n_failure_vectors': len(X_failure)}

    print(f'{layer:5s} | d={d:4d} | '
        f'success vectors={len(X_success):6d} | '
        f'failure vectors={len(X_failure):6d} | '
        f'quota={quota:.6f}')

# Rank candidate layers
ranking = sorted(results.items(), key=lambda item: item[1]['quota'], reverse=True)
print('\nLayer ranking')
for rank, (layer, info) in enumerate(ranking, start=1):
    print(f'{rank}. {layer}: quota={info["quota"]:.6f}')
print('\nSelected layer:', ranking[0][0])


down1 | d= 512 | success vectors=  9500 | failure vectors= 37500 | quota=0.064798
mid   | d=1024 | success vectors=  9500 | failure vectors= 37500 | quota=0.150553
up0   | d= 512 | success vectors=  9500 | failure vectors= 37500 | quota=0.160022
up1   | d= 256 | success vectors=  9500 | failure vectors= 37500 | quota=0.120938

Layer ranking
1. up0: quota=0.160022
2. mid: quota=0.150553
3. up1: quota=0.120938
4. down1: quota=0.064798

Selected layer: up0


In [20]:
layer = 'up0'
alphas = [0.1, 0.5, 1, 2, 10]

# Load all fitting episodes
train_episodes = []

for path in sorted(glob.glob(
    os.path.join(save_dir, 'steering_train_act_*.pkl'))):
    with open(path, 'rb') as f:
        train_episodes.extend(pickle.load(f))

# Collect pooled activation vectors by outcome
success_vectors = []
failure_vectors = []

for ep in train_episodes:
    act_path = os.path.join(save_dir, 'activations', f"episode_{ep['seed']}.npz")
    with np.load(act_path) as data:
        arr = data[layer].astype(np.float64)

    # Mean-pool over temporal action positions
    pooled = arr.mean(axis=-1)

    # Combine policy calls and denoising steps
    vectors = pooled.reshape(-1, pooled.shape[-1])

    if ep['success']:
        success_vectors.append(vectors)
    else:
        failure_vectors.append(vectors)

X_success = np.concatenate(success_vectors, axis=0)
X_failure = np.concatenate(failure_vectors, axis=0)


# Fit a conceptor from activation vectors
def compute_conceptor(X, alpha):
    X = X - X.mean(axis=0, keepdims=True)
    R = (X.T @ X) / len(X)
    eigenvalues, eigenvectors = np.linalg.eigh(R)
    weights = eigenvalues / (eigenvalues + alpha**-2)
    C = (eigenvectors * weights) @ eigenvectors.T
    return C

# Calculate success-failure overlap for each alpha
results = []

for alpha in alphas:
    C_success = compute_conceptor(X_success, alpha)
    C_failure = compute_conceptor(X_failure, alpha)
    numerator = np.trace(C_success @ C_failure)
    denominator = np.sqrt(np.trace(C_success @ C_success) * np.trace(C_failure @ C_failure))
    overlap = numerator / denominator
    results.append({
        'Alpha': alpha,
        'Overlap': overlap,
        'In target range': 0.85 <= overlap <= 0.95})

results_df = pd.DataFrame(results)

display(results_df.round({'Overlap': 4}))

# Select alpha values according to the paper's criterion
valid = results_df[results_df['In target range']]['Alpha'].tolist()

if valid:
    print('Selected alpha values:', valid)
else:
    distances = np.maximum(
        0.85 - results_df['Overlap'],
        results_df['Overlap'] - 0.95
    ).clip(lower=0)

    best_idx = distances.argmin()
    print('No alpha in target range')
    print('Closest alpha:', results_df.iloc[best_idx]['Alpha'])

,Alpha,Overlap,In target range
0,0.1,0.8977,True
1,0.5,0.8341,False
2,1.0,0.8245,False
3,2.0,0.8266,False
4,10.0,0.9148,True


Selected alpha values: [0.1, 10.0]


0.1 очень узкий концептор, пропускающий небольшое число направлений.  
10 широкий концептор, охватывающий значительную часть
пространства.  
  
Строим contrastive conceptors для двух оставленных значений alpha

In [21]:
alphas = [0.1, 10.0]

d = X_success.shape[1]
I = np.eye(d)

steering_conceptors = {}

for alpha in alphas:
    # Success and failure conceptors
    C_success = compute_conceptor(X_success, alpha)
    C_failure = compute_conceptor(X_failure, alpha)

    # Contrastive conceptor: success AND NOT failure
    C_steer = np.linalg.pinv(np.linalg.pinv(C_success) + np.linalg.pinv(I - C_failure) - I)
    steering_conceptors[alpha] = C_steer
    eigvals = np.linalg.eigvalsh(C_steer)

    print(f'Alpha = {alpha}')
    print(f'Dimension: {d}')
    print(f'Quota: {np.trace(C_steer) / d:.6f}')
    print(f'Min eigenvalue: {eigvals.min():.6e}')
    print(f'Max eigenvalue: {eigvals.max():.6f}')
    print()

Alpha = 0.1
Dimension: 512
Quota: 0.003166
Min eigenvalue: 7.358393e-06
Max eigenvalue: 0.250521

Alpha = 10.0
Dimension: 512
Quota: 0.160022
Min eigenvalue: 3.219562e-04
Max eigenvalue: 0.446099



# 4) И делаем стиринг в двух вариантах: простой contrastive вариант (например, что-то типа difference-in-means) и COAST-style steering.

In [22]:
# Load all completed test episodes
test_episodes = []

test_files = sorted(glob.glob(os.path.join(save_dir, 'baseline_test_*.pkl')))

for path in test_files:
    with open(path, 'rb') as f:
        test_episodes.extend(pickle.load(f))

test_seeds = sorted(ep['seed'] for ep in test_episodes)

print('Test episodes:', len(test_episodes))
print('Unique seeds:', len(set(test_seeds)))
print('Test seeds:', test_seeds)

Test episodes: 35
Unique seeds: 35
Test seeds: [300000, 300001, 300002, 300003, 300004, 300005, 300006, 300007, 300008, 300009, 300010, 300011, 300012, 300013, 300014, 300015, 300016, 300017, 300018, 300019, 300020, 300021, 300022, 300023, 300024, 300025, 300026, 300027, 300028, 300029, 300030, 300031, 300032, 300033, 300034]


In [23]:
# We test two alpha and two beta, selected according to the COAST protocol

alphas = [0.1, 10.0]
betas = [0.1, 0.3]

# Store steering matrices for all configurations
# M = (1 - beta) * I + beta * C_steer

steering_matrices = {}

for alpha in alphas:
    C_steer = steering_conceptors[alpha]

    d = C_steer.shape[0]
    I = np.eye(d)

    for beta in betas:
        M = (1 - beta) * I + beta * C_steer
        steering_matrices[(alpha, beta)] = torch.tensor(M, dtype=torch.float32)

print('Prepared steering configurations:')
for alpha, beta in steering_matrices:
    print(f'alpha={alpha}, beta={beta}, '
        f'matrix shape={tuple(steering_matrices[(alpha, beta)].shape)}')

# The hook intercepts the output of the selected U-Net block
# and transforms its channel representations using M

# Original output shape (B, C, T)
# B = batch size
# C = number of channels (512 for up0)
# T = temporal positions

# The same transformation is applied independently
# to every temporal position

def make_steering_hook(M):

    def steering_hook(module, inputs, output):

        if not torch.is_tensor(output):
            raise TypeError('Expected a tensor output.')

        if output.ndim != 3:
            raise ValueError(f'Expected output shape (B, C, T), '
                f'got {tuple(output.shape)}')

        if output.shape[1] != M.shape[0]:
            raise ValueError(f'Channel mismatch: output has {output.shape[1]} '
                f'channels, steering matrix has {M.shape[0]}')
        M_device = M.to(device=output.device, dtype=output.dtype)

        # Apply the steering transformation
        # h_steered = M @ h
        steered_output = torch.einsum('ij,bjt->bit', M_device, output)
        return steered_output
    return steering_hook

Prepared steering configurations:
alpha=0.1, beta=0.1, matrix shape=(512, 512)
alpha=0.1, beta=0.3, matrix shape=(512, 512)
alpha=10.0, beta=0.1, matrix shape=(512, 512)
alpha=10.0, beta=0.3, matrix shape=(512, 512)


In [24]:
np.savez(
    os.path.join(save_dir, 'normalization_stats.npz'),
    obs_min=stats['obs']['min'],
    obs_max=stats['obs']['max'],
    action_min=stats['action']['min'],
    action_max=stats['action']['max'])

# Save evaluation results after every rollout

def save_evaluation_results(results, filename):
    path = os.path.join(save_dir, filename)
    temp_path = path + '.tmp'

    with open(temp_path, 'wb') as f:
        pickle.dump(results, f)

    os.replace(temp_path, path)

In [25]:
# Create hook for Difference-in-Means

def make_mean_steering_hook(steering_vector, gamma):
    def steering_hook(module, inputs, output):
        v = torch.as_tensor(steering_vector, device=output.device, dtype=output.dtype)
        return output + gamma * v[None, :, None]
    return steering_hook

success_acts = []
failure_acts = []

for ep in train_episodes:
    seed = ep['seed']
    path = os.path.join(save_dir, 'activations', f'episode_{seed}.npz')

    with np.load(path) as data:
        acts = data['up0'].mean(axis=-1)

    episode_mean = acts.mean(axis=(0, 1))
    if ep['success'] == 1:
        success_acts.append(episode_mean)
    else:
        failure_acts.append(episode_mean)

mu_success = np.mean(success_acts, axis=0)
mu_failure = np.mean(failure_acts, axis=0)
steering_vector = mu_success - mu_failure

In [27]:
#@markdown ### **Difference-in-Means Steering Evaluation**

gammas = [0.5, 1.0, 2.0]
results_mean_steering = []

for gamma in gammas:
    print(f'\nGamma = {gamma}')

    for seed in test_seeds:
        result = run_episode(seed=seed, steering_config=('mean', gamma))
        results_mean_steering.append({
            'seed': seed,
            'gamma': gamma,
            'success': result['success'],
            'steps': result['steps'],
            'steps_to_success': result['steps_to_success'],
            'max_coverage': np.max(result['coverage'])})
        save_evaluation_results(results_mean_steering, 'results_mean_steering.pkl')

    gamma_results = [
        r for r in results_mean_steering
        if r['gamma'] == gamma]

    successes = sum(r['success'] for r in gamma_results)
    print(f'Success rate: {successes}/{len(gamma_results)}')


Gamma = 0.5


Success rate: 8/35

Gamma = 1.0
Success rate: 10/35

Gamma = 2.0


RuntimeError: Unable to find a valid cuDNN algorithm to run convolution

In [39]:
# Difference-in-Means Steering: resume evaluation

gamma = 2.0

completed_seeds = {int(r['seed'])
    for r in results_mean_steering
    if float(r['gamma']) == gamma}

remaining_seeds = [seed for seed in test_seeds
    if int(seed) not in completed_seeds]

print(f'Уже готово: {len(completed_seeds)}/35')
print(f'Осталось: {len(remaining_seeds)}')

for seed in remaining_seeds:
    result = run_episode(seed=seed, steering_config=('mean', gamma))
    results_mean_steering.append({
        'seed': seed,
        'gamma': gamma,
        'success': result['success'],
        'steps': result['steps'],
        'steps_to_success': result['steps_to_success'],
        'max_coverage': np.max(result['coverage'])})
    save_evaluation_results(
        results_mean_steering,
        'results_mean_steering.pkl')

    print(f'Seed {seed}: success={result["success"]}')

gamma_results = [r for r in results_mean_steering
    if float(r['gamma']) == gamma]

successes = sum(r['success'] for r in gamma_results)

print(f'\nGamma = {gamma}')
print(f'Success rate: {successes}/{len(gamma_results)}')

from collections import Counter

print('Всего эпизодов:', len(results_mean_steering))
print('По gamma:', Counter(
    float(r['gamma']) for r in results_mean_steering))

Уже готово: 16/35
Осталось: 19


Seed 300016: success=0
Seed 300017: success=0
Seed 300018: success=0
Seed 300019: success=0
Seed 300020: success=0
Seed 300021: success=1
Seed 300022: success=0
Seed 300023: success=0
Seed 300024: success=0
Seed 300025: success=0
Seed 300026: success=0
Seed 300027: success=0
Seed 300028: success=0
Seed 300029: success=0
Seed 300030: success=0
Seed 300031: success=0
Seed 300032: success=0
Seed 300033: success=0
Seed 300034: success=0

Gamma = 2.0
Success rate: 3/35
Всего эпизодов: 105
По gamma: Counter({0.5: 35, 1.0: 35, 2.0: 35})


In [42]:
#@markdown ### **COAST Steering Evaluation**

coast_configs = [(0.1, 0.1), (0.1, 0.3), (10.0, 0.1), (10.0, 0.3)]

results_coast = []

for alpha, beta in coast_configs:
    print(f'\nAlpha = {alpha}, Beta = {beta}')

    for seed in test_seeds:
        result = run_episode(seed=seed, steering_config=(alpha, beta))

        results_coast.append({
            'seed': seed,
            'alpha': alpha,
            'beta': beta,
            'success': result['success'],
            'steps': result['steps'],
            'steps_to_success': result['steps_to_success'],
            'max_coverage': np.max(result['coverage'])})
        save_evaluation_results(results_coast, 'results_coast.pkl')

    config_results = [r for r in results_coast if r['alpha'] == alpha and r['beta'] == beta]
    successes = sum(r['success'] for r in config_results)
    print(f'Success rate: {successes}/{len(config_results)}')


Alpha = 0.1, Beta = 0.1
Success rate: 13/35

Alpha = 0.1, Beta = 0.3
Success rate: 5/35

Alpha = 10.0, Beta = 0.1


RuntimeError: Unable to find a valid cuDNN algorithm to run convolution

In [27]:
results_path = 'results_coast.pkl'

# Load previously saved results
with open(results_path, 'rb') as f:
    results_coast = pickle.load(f)

coast_configs = [(0.1, 0.1), (0.1, 0.3), (10.0, 0.1), (10.0, 0.3)]

for alpha, beta in coast_configs:

    print(f'\nAlpha={alpha}, Beta={beta}')

    completed_seeds = {r['seed']
        for r in results_coast
        if r['alpha'] == alpha and r['beta'] == beta}

    for seed in test_seeds:
        if seed in completed_seeds:
            print(f'Seed {seed}: already completed, skipping')
            continue

        print(f'Running seed {seed}...')

        result = run_episode(seed=seed,
            steering_config=(alpha, beta))

        results_coast.append({
            'seed': seed,
            'alpha': alpha,
            'beta': beta,
            'success': result['success'],
            'steps': result['steps'],
            'steps_to_success': result['steps_to_success'],
            'max_coverage': np.max(result['coverage'])})

        save_evaluation_results(
            results_coast,
            results_path)

        completed_seeds.add(seed)

        del result
        gc.collect()

    config_results = [
        r for r in results_coast
        if r['alpha'] == alpha and r['beta'] == beta]

    successes = sum(r['success'] for r in config_results)

    print(
        f'Completed: {len(config_results)}/35 | '
        f'Successes: {successes}')

print('\nCOAST evaluation completed')



Alpha=0.1, Beta=0.1
Seed 300000: already completed, skipping
Seed 300001: already completed, skipping
Seed 300002: already completed, skipping
Seed 300003: already completed, skipping
Seed 300004: already completed, skipping
Seed 300005: already completed, skipping
Seed 300006: already completed, skipping
Seed 300007: already completed, skipping
Seed 300008: already completed, skipping
Seed 300009: already completed, skipping
Seed 300010: already completed, skipping
Seed 300011: already completed, skipping
Seed 300012: already completed, skipping
Seed 300013: already completed, skipping
Seed 300014: already completed, skipping
Seed 300015: already completed, skipping
Seed 300016: already completed, skipping
Seed 300017: already completed, skipping
Seed 300018: already completed, skipping
Seed 300019: already completed, skipping
Seed 300020: already completed, skipping
Seed 300021: already completed, skipping
Seed 300022: already completed, skipping
Seed 300023: already completed, skip

Running seed 300023...
Running seed 300024...
Running seed 300025...
Running seed 300026...
Running seed 300027...
Running seed 300028...
Running seed 300029...
Running seed 300030...
Running seed 300031...
Running seed 300032...
Running seed 300033...
Running seed 300034...
Completed: 35/35 | Successes: 14

Alpha=10.0, Beta=0.3
Running seed 300000...
Running seed 300001...
Running seed 300002...
Running seed 300003...
Running seed 300004...
Running seed 300005...
Running seed 300006...
Running seed 300007...
Running seed 300008...
Running seed 300009...
Running seed 300010...
Running seed 300011...
Running seed 300012...
Running seed 300013...
Running seed 300014...
Running seed 300015...
Running seed 300016...
Running seed 300017...
Running seed 300018...
Running seed 300019...
Running seed 300020...
Running seed 300021...
Running seed 300022...
Running seed 300023...
Running seed 300024...
Running seed 300025...
Running seed 300026...
Running seed 300027...
Running seed 300028...
Ru

# 5) Проверить и сравнить результат на небольшом наборе стартовых состояний


In [34]:
# Load all completed baseline test episodes
baseline_test_episodes = []

test_files = sorted(glob.glob(os.path.join(save_dir, 'baseline_test_*.pkl')))

for path in test_files:
    with open(path, 'rb') as f:
        baseline_test_episodes.extend(pickle.load(f))

print('Number of baseline episodes:', len(baseline_test_episodes))
print('Number of baseline episodes:', len(baseline_test_episodes))
print('Episode keys:', baseline_test_episodes[0].keys())
print('Example seed:', baseline_test_episodes[0]['seed'])
print('Example success:', baseline_test_episodes[0]['success'])
print('Example steps:', baseline_test_episodes[0]['steps'])
print('Coverage shape:', baseline_test_episodes[0]['coverage'].shape)
print('Max coverage:', np.max(baseline_test_episodes[0]['coverage']))

Number of baseline episodes: 35
Number of baseline episodes: 35
Episode keys: dict_keys(['seed', 'success', 'steps', 'steps_to_success', 'obs', 'actions', 'reward', 'coverage', 'call_pred', 'call_seeds', 'call_obs_indices'])
Example seed: 300000
Example success: 0
Example steps: 200
Coverage shape: (200,)
Max coverage: 0.9466141781688783


In [29]:
# Prepare baseline results for comparison

results_baseline = []

for ep in baseline_test_episodes:
    results_baseline.append({
        'seed': ep['seed'],
        'success': ep['success'],
        'steps': ep['steps'],
        'steps_to_success': ep['steps_to_success'],
        'max_coverage': np.max(ep['coverage'])})

save_evaluation_results(results_baseline, 'results_baseline.pkl')

successes = sum(r['success'] for r in results_baseline)

print(f'Baseline success rate: {successes}/{len(results_baseline)}')
print(f'Mean max coverage: {np.mean([r["max_coverage"] for r in results_baseline]):.4f}')

Baseline success rate: 10/35
Mean max coverage: 0.8865


In [33]:
# Prepare a summary table for all evaluated methods

all_results = {
    'Baseline': results_baseline,
    'Difference-in-Means': results_mean_steering,
    'COAST': results_coast}
summary = []

for method, results in all_results.items():
    # Group results by steering configuration
    configs = {}

    for r in results:
        if method == 'Baseline':
            config = 'No steering'
        elif method == 'Difference-in-Means':
            config = f"gamma={r['gamma']}"
        else:
            config = f"alpha={r['alpha']}, beta={r['beta']}"

        configs.setdefault(config, []).append(r)

    for config, episodes in configs.items():
        summary.append({
            'Method': method,
            'Configuration': config,
            'Success Rate': np.mean([r['success'] for r in episodes]),
            'Mean Steps': np.mean([r['steps'] for r in episodes]),
            'Mean Steps to Success': np.mean([
                r['steps_to_success'] for r in episodes
                if r['success'] == 1
            ]) if any(r['success'] == 1 for r in episodes) else np.nan,
            'Mean Max Coverage': np.mean([
                r['max_coverage'] for r in episodes])})

summary_df = pd.DataFrame(summary)
display(summary_df.round(4))

,Method,Configuration,Success Rate,Mean Steps,Mean Steps to Success,Mean Max Coverage
0,Baseline,No steering,0.2857,188.0286,158.1000,0.8865
1,Difference-in-Means,gamma=0.5,0.2286,188.0571,147.7500,0.8773
2,Difference-in-Means,gamma=1.0,0.2857,181.0286,133.6000,0.9016
3,Difference-in-Means,gamma=2.0,0.0857,194.9429,141.0000,0.8772
4,COAST,"alpha=0.1, beta=0.1",0.3714,179.0571,143.6154,0.9176
5,COAST,"alpha=0.1, beta=0.3",0.1429,193.6000,155.2000,0.7730
6,COAST,"alpha=10.0, beta=0.1",0.4000,176.7429,141.8571,0.9253
7,COAST,"alpha=10.0, beta=0.3",0.1429,190.9714,136.8000,0.7054


In [ ]:
# Ensure all result sets are loaded
import pickle

with open('results_baseline.pkl', 'rb') as f:
    results_baseline = pickle.load(f)

with open('results_mean_steering.pkl', 'rb') as f:
    results_mean_steering = pickle.load(f)

with open('results_coast.pkl', 'rb') as f:
    results_coast = pickle.load(f)


def compare_with_baseline(method_results, baseline_results):
    baseline = {r['seed']: bool(r['success']) for r in baseline_results}
    method = {r['seed']: bool(r['success']) for r in method_results}

    assert baseline.keys() == method.keys(), "Seed sets do not match!"

    fixed = sum(not baseline[s] and method[s] for s in baseline)
    broken = sum(baseline[s] and not method[s] for s in baseline)
    unchanged_success = sum(baseline[s] and method[s] for s in baseline)
    unchanged_failure = sum(not baseline[s] and not method[s] for s in baseline)

    return {
        'fixed': fixed,
        'broken': broken,
        'unchanged_success': unchanged_success,
        'unchanged_failure': unchanged_failure}


# Compare DiM configurations
for gamma in [0.5, 1.0, 2.0]:
    rows = [r for r in results_mean_steering if r['gamma'] == gamma]
    print(f"DiM gamma={gamma}:",
          compare_with_baseline(rows, results_baseline))

# Compare COAST configurations
for alpha, beta in [(0.1, 0.1), (0.1, 0.3), (10.0, 0.1), (10.0, 0.3)]:
    rows = [r for r in results_coast if r['alpha'] == alpha and r['beta'] == beta]
    print(f"COAST alpha={alpha}, beta={beta}:",
          compare_with_baseline(rows, results_baseline))

DiM gamma=0.5: {'fixed': 7, 'broken': 9, 'unchanged_success': 1, 'unchanged_failure': 18}
DiM gamma=1.0: {'fixed': 7, 'broken': 7, 'unchanged_success': 3, 'unchanged_failure': 18}
DiM gamma=2.0: {'fixed': 1, 'broken': 8, 'unchanged_success': 2, 'unchanged_failure': 24}
COAST alpha=0.1, beta=0.1: {'fixed': 8, 'broken': 5, 'unchanged_success': 5, 'unchanged_failure': 17}
COAST alpha=0.1, beta=0.3: {'fixed': 5, 'broken': 10, 'unchanged_success': 0, 'unchanged_failure': 20}
COAST alpha=10.0, beta=0.1: {'fixed': 9, 'broken': 5, 'unchanged_success': 5, 'unchanged_failure': 16}
COAST alpha=10.0, beta=0.3: {'fixed': 4, 'broken': 9, 'unchanged_success': 1, 'unchanged_failure': 21}
